In [1]:
# === Held-out evaluation for latest v3 run (macro/micro Dice, CSV+JSON) ===
from pathlib import Path
import importlib.util, json, time
import numpy as np

# --------- Paths (v3 run + held-out set) ----------
RUN_DIR   = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3/runs/20251110_101813")
TEST_DIR  = Path("/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/test_hires")
T1_DIR    = TEST_DIR / "t1"      # use these to avoid duplicate pairing
MSK_DIR   = TEST_DIR / "masks"
TRAIN_MOD = Path("/home/rbielski/stroke_cleaned/stroke_segmentation_v1.2/stroke_seg_v1.2_train.py")
# -----------------------------------------------

# ---- Import training module (utils + custom layers) ----
spec = importlib.util.spec_from_file_location("arc_seg_train", TRAIN_MOD)
seg = importlib.util.module_from_spec(spec)
spec.loader.exec_module(seg)

# ---- Find model to load (prefer full .keras; else rebuild and load weights) ----
models_dir     = RUN_DIR / "models"
callbacks_dir  = RUN_DIR / "callbacks"
full_models    = sorted(models_dir.glob("*.keras"))
best_weights   = callbacks_dir / "best_model_dynamic.weights.h5"
cfg_json_path  = models_dir / "config.json"  # written by training code

model = None
INPUT_SHAPE = None

custom_objects = {
    "ResidualConvBlock": seg.ResidualConvBlock,
    "VisionMambaBlock": seg.VisionMambaBlock,
    "SAM2Attention": seg.SAM2Attention,
    "CombinedLoss": seg.CombinedLoss,
    "dice_coefficient": seg.dice_coefficient,
    "dice_loss": seg.dice_loss,
    "boundary_loss": seg.boundary_loss,
}

try:
    from keras.saving import load_model as keras_load_model
except Exception:
    from tensorflow.keras.models import load_model as keras_load_model

if full_models:
    model_path = full_models[-1]
    print(f"Loading FULL model: {model_path}")
    model = keras_load_model(model_path, compile=False, custom_objects=custom_objects)
    INPUT_SHAPE = tuple(model.input_shape[1:])
else:
    assert cfg_json_path.exists(), f"Missing {cfg_json_path}"
    with open(cfg_json_path) as f:
        saved_cfg = json.load(f)
    cfg = seg.DynamicTrainingConfig(
        DATA_DIR=TEST_DIR,  # dummy; not training now
        MODEL_DIR=models_dir,
        CALLBACKS_DIR=callbacks_dir,
        INPUT_SHAPE=tuple(saved_cfg["INPUT_SHAPE"]) if saved_cfg.get("INPUT_SHAPE") else None,
        BASE_FILTERS=int(saved_cfg.get("BASE_FILTERS", 8)),
        SAM_HEADS=int(saved_cfg.get("SAM_HEADS", 2)),
    )
    if cfg.INPUT_SHAPE in (None, (), []):
        raise RuntimeError("INPUT_SHAPE missing in saved config; cannot rebuild model.")
    INPUT_SHAPE = cfg.INPUT_SHAPE
    print("Rebuilding model from config.json and loading best weights…")
    model = seg.build_dynamic_model(cfg)
    model.load_weights(str(best_weights))

print("INPUT_SHAPE:", INPUT_SHAPE)

# ---- Build held-out list using separated subfolders (avoid duplicates) ----
if T1_DIR.exists() and MSK_DIR.exists():
    cfg_eval = seg.DynamicTrainingConfig(DATA_DIR=TEST_DIR, IMAGES_DIR=T1_DIR, MASKS_DIR=MSK_DIR,
                                         MODEL_DIR=RUN_DIR/"_tmp_models", CALLBACKS_DIR=RUN_DIR/"_tmp_callbacks")
else:
    cfg_eval = seg.DynamicTrainingConfig(DATA_DIR=TEST_DIR,
                                         MODEL_DIR=RUN_DIR/"_tmp_models", CALLBACKS_DIR=RUN_DIR/"_tmp_callbacks")

cfg_eval.INPUT_SHAPE = INPUT_SHAPE
pairs, lesion_presence = seg.load_generic_dataset(cfg_eval)
print(f"Pairs: {len(pairs)} | % non-empty masks: {lesion_presence.mean()*100:.1f}%")

# ---- Dice helpers ----
def dice_soft(y, p):
    y = y.astype(np.float64); p = p.astype(np.float64)
    inter = (y * p).sum()
    return (2.0*inter) / (y.sum() + p.sum() + 1e-12)

def dice_hard(y, p, th=0.5):
    pb = (p >= th).astype(np.float64)
    inter = (y*pb).sum()
    return (2.0*inter) / (y.sum() + pb.sum() + 1e-12)

# ---- Evaluate (macro: per-case mean; micro: global) ----
macro_softs, macro_hards = [], []
s_inter_soft = np.float64(0.0)
s_sumy_soft  = np.float64(0.0)
s_sump_soft  = np.float64(0.0)

TH = 0.50  # default; you can sweep later

for i, (img_p, msk_p) in enumerate(pairs, 1):
    img = seg._load_and_preprocess_image(str(img_p), INPUT_SHAPE[:-1]).astype(np.float32)
    y   = seg._load_and_preprocess_mask(str(msk_p),  INPUT_SHAPE[:-1]).astype(np.float32)

    x = np.zeros((1,*INPUT_SHAPE), np.float32)
    x[0,...,0] = img
    p = model.predict(x, verbose=0)[0,...,0].astype(np.float32)

    ds = dice_soft(y, p)
    dh = dice_hard(y, p, th=TH)
    macro_softs.append(ds); macro_hards.append(dh)

    s_inter_soft += (y.astype(np.float64) * p.astype(np.float64)).sum()
    s_sumy_soft  += y.sum(dtype=np.float64)
    s_sump_soft  += p.sum(dtype=np.float64)

    if i % 10 == 0 or i == len(pairs):
        print(f"[{i}/{len(pairs)}] last case soft={ds:.4f} hard@{TH:.2f}={dh:.4f}")

macro_soft = float(np.mean(macro_softs))
macro_hard = float(np.mean(macro_hards))
micro_soft = float((2.0*s_inter_soft) / (s_sumy_soft + s_sump_soft + 1e-12))

# A second pass for exact micro-hard at the same threshold (memory-safe)
s_inter_hard = np.float64(0.0)
s_sumy_hard  = np.float64(0.0)
s_sump_hard  = np.float64(0.0)
for img_p, msk_p in pairs:
    y = seg._load_and_preprocess_mask(str(msk_p), INPUT_SHAPE[:-1]).astype(np.float32)
    x = np.zeros((1,*INPUT_SHAPE), np.float32)
    x[0,...,0] = seg._load_and_preprocess_image(str(img_p), INPUT_SHAPE[:-1]).astype(np.float32)
    p = model.predict(x, verbose=0)[0,...,0]
    pb = (p >= TH).astype(np.float64)
    s_inter_hard += (y.astype(np.float64) * pb).sum()
    s_sumy_hard  += y.sum(dtype=np.float64)
    s_sump_hard  += pb.sum(dtype=np.float64)

micro_hard = float((2.0*s_inter_hard) / (s_sumy_hard + s_sump_hard + 1e-12))

# ---- Report + save ----
print("\n=== HELD-OUT RESULTS ===")
print(f"Per-case (macro) soft Dice       : {macro_soft:.4f}")
print(f"Per-case (macro) hard Dice @ {TH:.2f}: {macro_hard:.4f}")
print(f"Global (micro) soft Dice         : {micro_soft:.4f}")
print(f"Global (micro) hard Dice @ {TH:.2f} : {micro_hard:.4f}")
print(f"Val set size: {len(pairs)} cases")

out_dir = RUN_DIR / "test_eval"
out_dir.mkdir(parents=True, exist_ok=True)
ts = time.strftime("%Y%m%d_%H%M%S")

# per-case CSV
csv_path = out_dir / f"test_metrics_{ts}.csv"
with open(csv_path, "w") as f:
    f.write("case,soft_dice,hard_dice_at_{:.2f}\n".format(TH))
    for (img_p, _), ds, dh in zip(pairs, macro_softs, macro_hards):
        f.write(f"{img_p.stem},{ds:.6f},{dh:.6f}\n")

# summary JSON
summary = {
    "threshold": TH,
    "macro_soft": macro_soft,
    "macro_hard": macro_hard,
    "micro_soft": micro_soft,
    "micro_hard": micro_hard,
    "n_cases": len(pairs),
    "run_dir": str(RUN_DIR),
}
json_path = out_dir / f"test_metrics_summary_{ts}.json"
with open(json_path, "w") as f:
    json.dump(summary, f, indent=2)

print("\nWrote per-case CSV ->", csv_path)
print("Wrote summary JSON ->", json_path)


2025-11-10 13:15:29.239283: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Visible GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')


I0000 00:00:1762805731.154436 1551677 gpu_process_state.cc:208] Using CUDA malloc Async allocator for GPU: 0
I0000 00:00:1762805731.155572 1551677 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22148 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:41:00.0, compute capability: 8.9
I0000 00:00:1762805731.155845 1551677 gpu_process_state.cc:208] Using CUDA malloc Async allocator for GPU: 1
I0000 00:00:1762805731.156919 1551677 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 22122 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:61:00.0, compute capability: 8.9
2025-11-10 13:15:31,218 - SmartSOTA_Dynamic - INFO - ✅ All imports successful
2025-11-10 13:15:31,219 - SmartSOTA_Dynamic - INFO - TensorFlow eager execution: True
2025-11-10 13:15:31,219 - SmartSOTA_Dynamic - INFO - Environment verified:
- Python 3.10.18 (main, Jun  5 2025, 13:14:17) [GCC 11.2.0]
- TensorFlo

Strategy: MirroredStrategy
Loading FULL model: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3/runs/20251110_101813/models/smart_sota_dynamic_20251110_101813.keras


2025-11-10 13:15:32,682 - SmartSOTA_Dynamic - INFO - 📚 Loading dataset (flex loader for T1w volumes)…
2025-11-10 13:15:32,683 - SmartSOTA_Dynamic - INFO - Memory at dataset_load_start: CPU=0.99GB | GPU mem tracking failed | Disk: 1230.2GB free
2025-11-10 13:15:32,688 - SmartSOTA_Dynamic - INFO - 📂 Two-folder mode: images=138 (/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/test_hires/t1), masks=138 (/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/test_hires/masks)
2025-11-10 13:15:32,689 - SmartSOTA_Dynamic - INFO - Found 138 image files and 138 mask files


INPUT_SHAPE: (192, 224, 192, 1)


2025-11-10 13:15:50,812 - SmartSOTA_Dynamic - INFO - 📊 Created 138 image–mask pairs
2025-11-10 13:15:50,813 - SmartSOTA_Dynamic - INFO - 🧠 Lesion presence: 100.00%
2025-11-10 13:15:50,813 - SmartSOTA_Dynamic - INFO - Memory at dataset_load_end: CPU=1.00GB | GPU mem tracking failed | Disk: 1230.2GB free


Pairs: 138 | % non-empty masks: 100.0%


2025-11-10 13:15:51.870160: I external/local_xla/xla/service/service.cc:163] XLA service 0x7bc92c005310 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-11-10 13:15:51.870185: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 4090, Compute Capability 8.9
2025-11-10 13:15:51.870191: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (1): NVIDIA GeForce RTX 4090, Compute Capability 8.9
2025-11-10 13:15:51.965566: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-11-10 13:15:52.240172: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91001
I0000 00:00:1762805758.458565 1551828 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


[10/138] last case soft=0.8617 hard@0.50=0.8619
[20/138] last case soft=0.8667 hard@0.50=0.8670
[30/138] last case soft=0.8474 hard@0.50=0.8476
[40/138] last case soft=0.5685 hard@0.50=0.5682
[50/138] last case soft=0.7938 hard@0.50=0.7943
[60/138] last case soft=0.8585 hard@0.50=0.8585
[70/138] last case soft=0.8423 hard@0.50=0.8431
[80/138] last case soft=0.6236 hard@0.50=0.6262
[90/138] last case soft=0.8067 hard@0.50=0.8073
[100/138] last case soft=0.0312 hard@0.50=0.0313
[110/138] last case soft=0.6186 hard@0.50=0.6209
[120/138] last case soft=0.5342 hard@0.50=0.5356
[130/138] last case soft=0.4264 hard@0.50=0.4294
[138/138] last case soft=0.0283 hard@0.50=0.0283

=== HELD-OUT RESULTS ===
Per-case (macro) soft Dice       : 0.5888
Per-case (macro) hard Dice @ 0.50: 0.5895
Global (micro) soft Dice         : 0.7460
Global (micro) hard Dice @ 0.50 : 0.7463
Val set size: 138 cases

Wrote per-case CSV -> /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3/runs/20251110_1

In [1]:
# === Interactive MRI viewer for v3 held-out eval (uses saved preds *or* computes on-the-fly) ===
# CONFIG — set these two:
RUN_DIR  = "/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3/runs/20251110_101813"
TEST_DIR = "/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/test_hires"

# Optional: if you already saved NIfTI preds, point here; else leave "" to compute on-the-fly.
PREDS_DIR = ""   # e.g. "/.../test_preds_t050"

# -----------------------------------------------------------------------------------------------
import os, re, glob, numpy as np, nibabel as nib, matplotlib.pyplot as plt, importlib.util, time, json, pathlib
from ipywidgets import Dropdown, IntSlider, FloatSlider, RadioButtons, Checkbox, Button, HBox, VBox, Layout, HTML, Output
from IPython.display import display

RUN_DIR   = pathlib.Path(RUN_DIR)
TEST_DIR  = pathlib.Path(TEST_DIR)
T1_DIR    = TEST_DIR / "t1"
MSK_DIR   = TEST_DIR / "masks"
PREDS_DIR = pathlib.Path(PREDS_DIR) if PREDS_DIR else None

# ---- load training module & model (for on-the-fly mode) ----
def _load_seg_module():
    TRAIN_MOD = "/home/rbielski/stroke_cleaned/stroke_segmentation_v1.2/stroke_seg_v1.2_train.py"
    spec = importlib.util.spec_from_file_location("arc_seg_train", TRAIN_MOD)
    seg  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(seg)
    return seg

def _load_model_if_needed():
    if PREDS_DIR and PREDS_DIR.exists():
        return None, None  # using precomputed NIfTIs; no model needed
    # load full .keras if present else rebuild from config.json + best weights
    seg = _load_seg_module()
    models_dir    = RUN_DIR / "models"
    callbacks_dir = RUN_DIR / "callbacks"
    full_models   = sorted(models_dir.glob("*.keras"))
    custom_objects = {
        "ResidualConvBlock": seg.ResidualConvBlock,
        "VisionMambaBlock": seg.VisionMambaBlock,
        "SAM2Attention": seg.SAM2Attention,
        "CombinedLoss": seg.CombinedLoss,
        "dice_coefficient": seg.dice_coefficient,
        "dice_loss": seg.dice_loss,
        "boundary_loss": seg.boundary_loss,
    }
    try:
        from keras.saving import load_model as keras_load_model
    except Exception:
        from tensorflow.keras.models import load_model as keras_load_model

    if full_models:
        m = keras_load_model(full_models[-1], compile=False, custom_objects=custom_objects)
        input_shape = tuple(m.input_shape[1:])
        return (seg, (m, input_shape))
    # fallback rebuild
    cfg_json = models_dir / "config.json"
    assert cfg_json.exists(), f"Missing {cfg_json}"
    with open(cfg_json) as f: saved_cfg = json.load(f)
    cfg = seg.DynamicTrainingConfig(
        DATA_DIR=TEST_DIR, MODEL_DIR=models_dir, CALLBACKS_DIR=callbacks_dir,
        INPUT_SHAPE=tuple(saved_cfg["INPUT_SHAPE"]) if saved_cfg.get("INPUT_SHAPE") else None,
        BASE_FILTERS=int(saved_cfg.get("BASE_FILTERS", 8)),
        SAM_HEADS=int(saved_cfg.get("SAM_HEADS", 2)),
    )
    assert cfg.INPUT_SHAPE, "INPUT_SHAPE missing in saved config; cannot rebuild"
    m = seg.build_dynamic_model(cfg)
    m.load_weights(str(callbacks_dir / "best_model_dynamic.weights.h5"))
    return (seg, (m, cfg.INPUT_SHAPE))

# ---- file matching helpers ----
def _norm_key(name: str) -> str:
    n = re.sub(r"\.nii(\.gz)?$", "", name)
    n = re.sub(r"_T1w", "", n)
    n = re.sub(r"_lesion_mask(_MNI)?_clean", "", n)
    n = re.sub(r"_MNI_norm", "", n)
    n = re.sub(r"_(soft|hard)$", "", n)
    return n

def _rglob(patterns, root):
    out = []
    for p in patterns:
        out.extend(glob.glob(os.path.join(str(root), "**", p), recursive=True))
    return sorted(out)

def _gather_gt_t1(test_dir):
    gt_map  = {_norm_key(os.path.basename(p)): p for p in _rglob(["*lesion_mask*.nii.gz"], test_dir)}
    mri_map = {_norm_key(os.path.basename(p)): p for p in _rglob(["*T1w*.nii.gz"], test_dir)}
    return gt_map, mri_map

def _gather_preds(preds_root):
    soft = _rglob(["*_soft.nii.gz"], preds_root)
    hard = _rglob(["*_hard.nii.gz"], preds_root)
    return ({_norm_key(os.path.basename(p)): p for p in soft},
            {_norm_key(os.path.basename(p)): p for p in hard})

# ---- build case list (supports: precomputed preds or on-the-fly) ----
seg_mod, model_bundle = _load_model_if_needed()
gt_map, mri_map = _gather_gt_t1(TEST_DIR)

if PREDS_DIR and PREDS_DIR.exists():
    psoft_map, phard_map = _gather_preds(PREDS_DIR)
    keys = sorted(set(psoft_map) & set(gt_map) & set(mri_map))
    cases = [{
        "key": k, "mri": mri_map[k], "gt": gt_map[k],
        "pred_soft": psoft_map[k], "pred_hard": phard_map.get(k)  # may be None
    } for k in keys]
else:
    # on-the-fly prediction cache
    CACHE_DIR = RUN_DIR / "test_view_cache"
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    psoft_map, phard_map = {}, {}
    keys = sorted(set(gt_map) & set(mri_map))
    cases = [{
        "key": k, "mri": mri_map[k], "gt": gt_map[k],
        "pred_soft": str(CACHE_DIR / f"{k}_soft.nii.gz"),
        "pred_hard": None
    } for k in keys]

if not cases:
    raise RuntimeError("No matching cases. Check TEST_DIR (and PREDS_DIR if set).")

# ---- lazy loaders / cache ----
_cache = {}
def _ensure_pred(case):
    """If on-the-fly mode, materialize *_soft.nii.gz to cache."""
    if PREDS_DIR and PREDS_DIR.exists():
        return  # already have files
    assert seg_mod is not None and model_bundle is not None, "Model not loaded"
    m, INPUT_SHAPE = model_bundle
    soft_path = pathlib.Path(case["pred_soft"])
    if soft_path.exists():
        return
    # compute prediction
    img = seg_mod._load_and_preprocess_image(case["mri"], INPUT_SHAPE[:-1]).astype(np.float32)
    x = np.zeros((1,*INPUT_SHAPE), np.float32); x[0,...,0] = img
    p = m.predict(x, verbose=0)[0,...,0].astype(np.float32)
    # write NIfTI aligned to MRI
    affine = nib.load(case["mri"]).affine
    nib.save(nib.Nifti1Image(p, affine), str(soft_path))

def load_case(case):
    k = case["key"]
    if k not in _cache:
        _ensure_pred(case)
        mri = np.asanyarray(nib.load(case["mri"]).get_fdata())
        gt  = np.asanyarray(nib.load(case["gt"]).get_fdata())
        ps  = np.asanyarray(nib.load(case["pred_soft"]).get_fdata())
        # pred_hard may be absent; we’ll derive from ps at chosen threshold
        ph  = np.asanyarray(nib.load(case["pred_hard"]).get_fdata()) if case.get("pred_hard") and os.path.exists(case["pred_hard"]) else None
        # crop to common shape
        min_shape = tuple(min(d) for d in zip(mri.shape, gt.shape, ps.shape))
        slc = tuple(slice(0,s) for s in min_shape)
        _cache[k] = (mri[slc], gt[slc], ps[slc], (ph[slc] if ph is not None else None))
    return _cache[k]

def perc_norm(x, pmin=0.5, pmax=99.5):
    a, b = np.percentile(x, [pmin, pmax])
    if b <= a: return np.zeros_like(x, np.float32)
    y = (x - a) / (b - a)
    return np.clip(y, 0, 1).astype(np.float32)

def plane(arr, axis, idx):
    return arr[:,:,idx] if axis==2 else (arr[:,idx,:] if axis==1 else arr[idx,:,:])

# ---- widgets ----
w_case   = Dropdown(options=[(c["key"], i) for i,c in enumerate(cases)], description="Case:", layout=Layout(width="45%"))
w_axis   = RadioButtons(options=[("axial (z)", 2), ("coronal (y)", 1), ("sagittal (x)", 0)], value=2, description="Axis:")
w_slice  = IntSlider(description="Slice:", min=0, max=10, value=0, step=1, layout=Layout(width="60%"))
w_overlay= RadioButtons(options=["None", "GT", "Pred (hard)", "Pred (soft)"], value="Pred (hard)", description="Overlay:")
w_alpha  = FloatSlider(description="Alpha:", min=0.0, max=1.0, value=0.35, step=0.05)
w_softth = FloatSlider(description="Soft Th:", min=0.0, max=1.0, value=0.50, step=0.01)  # default 0.50 for v3
w_norm   = Checkbox(value=True, description="Normalize MRI (0.5–99.5%)")
w_redraw = Button(description="Redraw", button_style="")
w_info   = HTML(value="")
out      = Output(layout=Layout(border="1px solid #333", width="720px", height="720px"))

def refresh_slice_range(*_):
    mri, gt, ps, ph = load_case(cases[w_case.value])
    w_slice.max = mri.shape[w_axis.value]-1
    if w_slice.value > w_slice.max:
        w_slice.value = w_slice.max

def redraw(*_):
    with out:
        out.clear_output(wait=True)
        case = cases[w_case.value]
        mri, gt, ps, ph = load_case(case)
        axc, sl = w_axis.value, w_slice.value

        bg = plane(mri, axc, sl)
        bg = perc_norm(bg) if w_norm.value else ((bg - bg.min()) / max(1e-6, bg.max()-bg.min()))

        ov = None; title_extra = ""
        if w_overlay.value == "GT":
            ov = plane((gt>0).astype(np.float32), axc, sl); title_extra = " | GT"
        elif w_overlay.value == "Pred (hard)":
            if ph is not None:
                ov = plane((ph>0).astype(np.float32), axc, sl); title_extra = " | Pred hard (file)"
            else:
                ov = (plane(ps, axc, sl) >= w_softth.value).astype(np.float32); title_extra = f" | Pred hard (soft≥{w_softth.value:.2f})"
        elif w_overlay.value == "Pred (soft)":
            ov = (plane(ps, axc, sl) >= w_softth.value).astype(np.float32)
            title_extra = f" | Pred soft ≥ {w_softth.value:.02f}"

        fig, ax = plt.subplots(figsize=(6,6))
        ax.imshow(bg.T, origin="lower", cmap="gray")
        if ov is not None:
            ax.imshow(ov.T, origin="lower", alpha=w_alpha.value)
            try: ax.contour(ov.T, levels=[0.5], colors=["r"], linewidths=0.8)
            except Exception: pass
        ax.set_title(f"{case['key']} | slice {sl}{title_extra}")
        ax.axis("off")
        plt.show()

def _on_change(_):
    refresh_slice_range()
    redraw()

for w in (w_case, w_axis): w.observe(_on_change, names="value")
for w in (w_slice, w_overlay, w_alpha, w_softth, w_norm): w.observe(redraw, names="value")
w_redraw.on_click(redraw)

refresh_slice_range(); redraw()
controls = VBox([
    HBox([w_case, w_axis]),
    HBox([w_slice]),
    HBox([w_overlay, w_alpha, w_softth, w_norm, w_redraw]),
    w_info
])
display(controls, out)


2025-11-10 13:28:14.287966: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Visible GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')


I0000 00:00:1762806495.935692 1571263 gpu_process_state.cc:208] Using CUDA malloc Async allocator for GPU: 0
I0000 00:00:1762806495.937005 1571263 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22148 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:41:00.0, compute capability: 8.9
I0000 00:00:1762806495.937355 1571263 gpu_process_state.cc:208] Using CUDA malloc Async allocator for GPU: 1
I0000 00:00:1762806495.938424 1571263 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 22122 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:61:00.0, compute capability: 8.9
2025-11-10 13:28:15,984 - SmartSOTA_Dynamic - INFO - ✅ All imports successful
2025-11-10 13:28:15,984 - SmartSOTA_Dynamic - INFO - TensorFlow eager execution: True
2025-11-10 13:28:15,985 - SmartSOTA_Dynamic - INFO - Environment verified:
- Python 3.10.18 (main, Jun  5 2025, 13:14:17) [GCC 11.2.0]
- TensorFlo

Strategy: MirroredStrategy


Output(layout=Layout(border_bottom='1px solid #333', border_left='1px solid #333', border_right='1px solid #33…

In [6]:
# === Robust join: handle eval CSV with either `case` or `key` ===
from pathlib import Path
import csv, re, math, time
import numpy as np
import nibabel as nib

# ---------- CONFIG ----------
RUN_DIR    = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3/runs/20251110_101813")
TEST_EVAL  = RUN_DIR / "test_eval"
# Use the plain per-case eval if you have it; otherwise this also supports the merged one:
EVAL_CSV   = TEST_EVAL / "test_metrics_20251110_131754.csv"  # <- prefer this
# If you only have the merged file, use that instead:
# EVAL_CSV = TEST_EVAL / "test_metrics_with_manifest_20251110_134855.csv"

MANIFEST   = Path("/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/combined_hires_lores_manifest.csv")
# -------------------------------------------

def _norm_key(name: str) -> str:
    n = re.sub(r"\.nii(\.gz)?$", "", name or "")
    n = re.sub(r"_T1w", "", n)
    n = re.sub(r"_MNI_norm", "", n)
    n = re.sub(r"_lesion_mask(_MNI)?_clean", "", n)
    n = re.sub(r"_(soft|hard)$", "", n)
    return n

def _to_float(x):
    try: return float(x)
    except Exception: return float("nan")

def _to_int(x):
    try: return int(float(x))
    except Exception: return 0

# --- Load manifest keyed by `key` ---
manifest = {}
with open(MANIFEST, newline="") as f:
    for row in csv.DictReader(f):
        k = (row.get("key") or "").strip()
        if not k:
            # derive from t1_path/mask_path if key missing
            for src in (row.get("t1_path"), row.get("mask_path")):
                if src:
                    k = _norm_key(Path(src).name)
                    break
        if not k:
            continue
        manifest[k] = {
            "dataset": row.get("dataset", ""),
            "t1_path": row.get("t1_path", ""),
            "mask_path": row.get("mask_path", ""),
            "fwhm_mm": _to_float(row.get("fwhm_mm", "")),
            "hi_freq_energy": _to_float(row.get("hi_freq_energy", "")),
            "lap_var": _to_float(row.get("lap_var", "")),
            "score": _to_float(row.get("score", "")),
            "bin": row.get("bin", ""),
            "mask_voxels": _to_int(row.get("mask_voxels", "")),
            "mask_ml": _to_float(row.get("mask_ml", "")),
            "mask_voxels_clean": _to_int(row.get("mask_voxels_clean", "")),
            "mask_ml_clean": _to_float(row.get("mask_ml_clean", "")),
            "brain_frac": _to_float(row.get("brain_frac", "")),
            "vox_total": _to_int(row.get("vox_total", "")),
            "norm_img_nonzero": _to_int(row.get("norm_img_nonzero", "")),
            "brain_voxels": _to_int(row.get("brain_voxels", "")),
            "voxel_mm3": _to_float(row.get("voxel_mm3", "")),
        }

print("Manifest rows:", len(manifest))

# --- Load eval CSV; accept either `case` or `key` ---
with open(EVAL_CSV, newline="") as f:
    rd = csv.DictReader(f)
    cols = [c for c in (rd.fieldnames or []) if c]
    has_key  = "key"  in cols
    has_case = "case" in cols
    # Find hard dice col: "hard_dice" or "hard_dice_at_*"
    hard_col = None
    for c in cols:
        if c == "hard_dice" or c.startswith("hard_dice_at_"):
            hard_col = c; break
    if hard_col is None:
        hard_col = "hard_dice_at_0.50"  # best guess

    eval_rows = []
    for row in rd:
        if has_key:
            k = (row.get("key") or "").strip()
        elif has_case:
            k = _norm_key(row.get("case") or "")
        else:
            # last-ditch: try to infer from any present path-like column
            candidates = [row.get(x,"") for x in cols if "path" in x.lower()]
            k = ""
            for cand in candidates:
                if cand:
                    k = _norm_key(Path(cand).name); break

        if not k:
            continue
        eval_rows.append({
            "key": k,
            "soft_dice": _to_float(row.get("soft_dice", "")),
            "hard_dice": _to_float(row.get(hard_col, "")),
        })

print("Eval rows:", len(eval_rows))

# --- Recompute lesion size when manifest is missing/zero ---
def _recompute_size(mask_path: str, voxel_mm3: float):
    try:
        m = np.asanyarray(nib.load(mask_path).get_fdata())
        vox = int((m > 0).sum())
        ml  = float(vox * (voxel_mm3 if not math.isnan(voxel_mm3) and voxel_mm3 > 0 else 1.0) / 1000.0)
        return vox, ml
    except Exception:
        return 0, float("nan")

joined, missing, recomputed = [], 0, 0
for r in eval_rows:
    info = manifest.get(r["key"])
    if not info:
        missing += 1
        continue
    vox = info["mask_voxels_clean"] or info["mask_voxels"] or 0
    ml  = info["mask_ml_clean"] if not math.isnan(info["mask_ml_clean"]) else info["mask_ml"]
    if vox == 0:
        v2, m2 = _recompute_size(info["mask_path"], info["voxel_mm3"])
        vox, ml = v2, m2
        recomputed += 1
    joined.append({
        "key": r["key"],
        "soft_dice": r["soft_dice"],
        "hard_dice": r["hard_dice"],
        "voxels_clean": int(vox),
        "ml_clean": float(ml) if (ml is not None and not math.isnan(ml)) else float("nan"),
        "fwhm_mm": info["fwhm_mm"],
        "hi_freq_energy": info["hi_freq_energy"],
        "lap_var": info["lap_var"],
        "score": info["score"],
        "brain_frac": info["brain_frac"],
        "vox_total": info["vox_total"],
        "dataset": info["dataset"],
        "bin": info["bin"],
        "mask_path": info["mask_path"],
        "voxel_mm3": info["voxel_mm3"],
    })

print(f"Joined cases: {len(joined)} | Missing in manifest: {missing} | Recomputed sizes: {recomputed}")

# If joins still look wrong, show a few keys that failed:
if len(joined) == 0 and missing > 0:
    eval_keys = {r["key"] for r in eval_rows}
    man_keys  = set(manifest.keys())
    not_in_manifest = sorted(list(eval_keys - man_keys))[:10]
    print("Examples not in manifest:", not_in_manifest)

# --- Correlations & summaries (same as before) ---
def _col(name, dtype=np.float64):
    return np.array([row[name] for row in joined], dtype=dtype)

def nan_pearson(x, y):
    m = (~np.isnan(x)) & (~np.isnan(y))
    if m.sum() < 3: return float("nan")
    return float(np.corrcoef(x[m], y[m])[0,1])

if joined:
    soft = _col("soft_dice"); hard = _col("hard_dice"); vox = _col("voxels_clean")
    fwhm = _col("fwhm_mm"); hfe = _col("hi_freq_energy"); lapv = _col("lap_var")
    score = _col("score"); bfr = _col("brain_frac")
    log_vox = np.log10(np.where(vox > 0, vox, np.nan))

    print("\n=== Correlations (Pearson r) ===")
    print(f"r(soft_dice, log10(lesion_vox)) = {nan_pearson(soft, log_vox):.3f}")
    print(f"r(hard_dice, log10(lesion_vox)) = {nan_pearson(hard, log_vox):.3f}")
    print(f"r(soft_dice, fwhm_mm)           = {nan_pearson(soft, fwhm):.3f}")
    print(f"r(soft_dice, hi_freq_energy)    = {nan_pearson(soft, hfe):.3f}")
    print(f"r(soft_dice, lap_var)           = {nan_pearson(soft, lapv):.3f}")
    print(f"r(soft_dice, score)             = {nan_pearson(soft, score):.3f}")
    print(f"r(soft_dice, brain_frac)        = {nan_pearson(soft, bfr):.3f}")

    bins = [0, 2500, 10000, 30000, 60000, 10**9]
    labels = ["[0,2.5k)", "[2.5k,10k)", "[10k,30k)", "[30k,60k)", "[60k,+inf)"]
    print("\n=== Size-bin summary (voxels_clean) ===")
    for i in range(len(bins)-1):
        lo, hi = bins[i], bins[i+1]
        m = (vox >= lo) & (vox < hi)
        n = int(m.sum())
        ms = float(np.nanmean(soft[m])) if n > 0 else float("nan")
        mh = float(np.nanmean(hard[m])) if n > 0 else float("nan")
        print(f"{labels[i]:>12} | n={n:3d} | mean soft={ms:.4f} | mean hard={mh:.4f}")

    # Save merged CSV
    out_dir = TEST_EVAL; out_dir.mkdir(parents=True, exist_ok=True)
    ts = time.strftime("%Y%m%d_%H%M%S")
    out_path = out_dir / f"test_metrics_with_manifest_FIXED_{ts}.csv"
    with open(out_path, "w", newline="") as f:
        wr = csv.DictWriter(f, fieldnames=list(joined[0].keys()))
        wr.writeheader(); wr.writerows(joined)
    print("\nSaved merged per-case CSV ->", out_path)


Manifest rows: 659
Eval rows: 138
Joined cases: 138 | Missing in manifest: 0 | Recomputed sizes: 138

=== Correlations (Pearson r) ===
r(soft_dice, log10(lesion_vox)) = 0.641
r(hard_dice, log10(lesion_vox)) = 0.639
r(soft_dice, fwhm_mm)           = nan
r(soft_dice, hi_freq_energy)    = -0.195
r(soft_dice, lap_var)           = 0.304
r(soft_dice, score)             = 0.019
r(soft_dice, brain_frac)        = nan

=== Size-bin summary (voxels_clean) ===
    [0,2.5k) | n= 28 | mean soft=0.2797 | mean hard=0.2809
  [2.5k,10k) | n= 20 | mean soft=0.5299 | mean hard=0.5310
   [10k,30k) | n= 32 | mean soft=0.6071 | mean hard=0.6076
   [30k,60k) | n= 34 | mean soft=0.7103 | mean hard=0.7107
  [60k,+inf) | n= 24 | mean soft=0.8020 | mean hard=0.8022

Saved merged per-case CSV -> /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3/runs/20251110_101813/test_eval/test_metrics_with_manifest_FIXED_20251110_160551.csv


/home/rbielski/miniconda3/envs/tf_310/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/rbielski/miniconda3/envs/tf_310/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:3046: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [7]:
# === Plot + partial correlations + top/bottom cases ===
from pathlib import Path
import csv, time, math
import numpy as np
import matplotlib.pyplot as plt

RUN_DIR   = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3/runs/20251110_101813")
CSV_PATH  = RUN_DIR / "test_eval" / "test_metrics_with_manifest_FIXED_20251110_160551.csv"
OUT_DIR   = RUN_DIR / "test_eval" / "figs"; OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- load
rows = []
with open(CSV_PATH, newline="") as f:
    rd = csv.DictReader(f)
    for r in rd:
        rows.append(r)

def col(name, cast=float):
    out = []
    for r in rows:
        v = r.get(name, "")
        try: out.append(cast(v))
        except: out.append(float("nan"))
    return np.array(out, dtype=float)

soft  = col("soft_dice")
hard  = col("hard_dice")
vox   = col("voxels_clean", int).astype(float)
hfe   = col("hi_freq_energy")
lapv  = col("lap_var")
score = col("score")
bfr   = col("brain_frac")
logv  = np.log10(np.where(vox>0, vox, np.nan))

def nan_corr(x, y):
    m = (~np.isnan(x)) & (~np.isnan(y))
    if m.sum() < 3: return np.nan
    return float(np.corrcoef(x[m], y[m])[0,1])

# partial corr(soft, metric | logv) via linear residuals
def partial_corr(soft, metric, control):
    m = (~np.isnan(soft)) & (~np.isnan(metric)) & (~np.isnan(control))
    if m.sum() < 5: return np.nan
    s, z, c = soft[m], metric[m], control[m]
    # regress out control: residual = x - proj(x|c)
    def resid(x, c):
        A = np.vstack([c, np.ones_like(c)]).T
        beta, *_ = np.linalg.lstsq(A, x, rcond=None)
        return x - A @ beta
    rs = resid(s, c); rz = resid(z, c)
    return nan_corr(rs, rz)

pcorr_hfe  = partial_corr(soft, hfe, logv)
pcorr_lapv = partial_corr(soft, lapv, logv)
pcorr_score= partial_corr(soft, score, logv)

print("Partial r (soft, hfe | log size)  :", f"{pcorr_hfe:.3f}")
print("Partial r (soft, lapv | log size):", f"{pcorr_lapv:.3f}")
print("Partial r (soft, score | log size):", f"{pcorr_score:.3f}")

# ---- scatter helpers
def scatter(x, y, xlabel, ylabel, fname):
    m = (~np.isnan(x)) & (~np.isnan(y))
    xs, ys = x[m], y[m]
    plt.figure(figsize=(5.2,4.0))
    plt.scatter(xs, ys, s=14, alpha=0.6)
    # least-squares line
    if xs.size >= 2:
        A = np.vstack([xs, np.ones_like(xs)]).T
        b, a = np.linalg.lstsq(A, ys, rcond=None)[0]
        xx = np.linspace(xs.min(), xs.max(), 100)
        plt.plot(xx, a + b*xx)
        r = np.corrcoef(xs, ys)[0,1]
        plt.title(f"r = {r:.3f}")
    plt.xlabel(xlabel); plt.ylabel(ylabel); plt.tight_layout()
    p = OUT_DIR / fname
    plt.savefig(p, dpi=150); plt.close()
    print("Saved", p)

scatter(logv, soft, "log10(lesion vox)", "soft Dice", "scatter_soft_vs_logvox.png")
scatter(hfe,  soft, "hi_freq_energy",   "soft Dice", "scatter_soft_vs_hfe.png")
scatter(lapv, soft, "lap_var",          "soft Dice", "scatter_soft_vs_lapvar.png")
scatter(score,soft, "score",            "soft Dice", "scatter_soft_vs_score.png")

# ---- top/bottom by soft Dice (with size)
keys  = [r["key"] for r in rows]
order = np.argsort(soft)  # ascending
bottom_idx = order[:10]
top_idx    = order[-10:][::-1]

def show_list(ix, title):
    print("\n"+title)
    for i in ix:
        print(f"  {keys[i]:40s} | soft={soft[i]:.4f} | vox={int(vox[i])}")

show_list(bottom_idx, "Bottom 10 by soft Dice")
show_list(top_idx,    "Top 10 by soft Dice")


Partial r (soft, hfe | log size)  : -0.075
Partial r (soft, lapv | log size): 0.018
Partial r (soft, score | log size): -0.071
Saved /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3/runs/20251110_101813/test_eval/figs/scatter_soft_vs_logvox.png
Saved /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3/runs/20251110_101813/test_eval/figs/scatter_soft_vs_hfe.png
Saved /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3/runs/20251110_101813/test_eval/figs/scatter_soft_vs_lapvar.png
Saved /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3/runs/20251110_101813/test_eval/figs/scatter_soft_vs_score.png

Bottom 10 by soft Dice
  sub-M2164_ses-622                        | soft=0.0000 | vox=5277
  sub-r042s003_ses-1                       | soft=0.0000 | vox=1311
  sub-r038s065_ses-1                       | soft=0.0000 | vox=1646
  sub-r048s006_ses-1                       | soft=0.0000 | vox=20338
  sub-r046s008_ses-1            

In [10]:
# === Write a compact Markdown report from the latest NON-EMPTY joined CSV ===
from pathlib import Path
from datetime import datetime
import csv, math, numpy as np, os

# Point at your v3 run folder
RUN_DIR  = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3/runs/20251110_101813")
EVAL_DIR = RUN_DIR / "test_eval"
FIG_DIR  = EVAL_DIR / "figs"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

def csv_row_count(p: Path) -> int:
    try:
        with open(p, newline="") as f:
            r = csv.reader(f)
            header = next(r, None)
            if header is None:
                return 0
            n = 0
            for _ in r:
                n += 1
            return n
    except Exception:
        return 0

# Prefer files containing "with_manifest"; pick the newest with >=1 data row
candidates = sorted(EVAL_DIR.glob("test_metrics_with_manifest*.csv"), key=lambda p: p.stat().st_mtime)
candidates = candidates[::-1]  # newest first

CSV_JOIN = None
for p in candidates:
    if csv_row_count(p) > 0:
        CSV_JOIN = p
        break

if CSV_JOIN is None:
    # Show diagnostic listing and stop early with a helpful message
    print("No non-empty 'with_manifest' CSVs found. Here are candidates with row counts:")
    for p in candidates[::-1]:
        print(f"- {p.name}  | rows={csv_row_count(p)}  | mtime={datetime.fromtimestamp(p.stat().st_mtime)}")
    raise SystemExit(0)

print("Using eval CSV:", CSV_JOIN)

# --- load rows ---
rows = []
with open(CSV_JOIN, newline="") as f:
    rd = csv.DictReader(f)
    for r in rd:
        rows.append(r)
if not rows:
    raise RuntimeError("CSV has no data rows (unexpected after the non-empty check).")

# --- pull numeric columns safely ---
def col(name, cast=float):
    arr = []
    for r in rows:
        v = r.get(name, "")
        try:
            arr.append(cast(v))
        except Exception:
            arr.append(float("nan"))
    return np.array(arr, dtype=float)

def pick(*names):
    """Return first available column among names."""
    for nm in names:
        if nm in rows[0]:
            return nm
    return names[-1]

soft_col = pick("soft_dice")
hard_col = pick("hard_dice", "hard_dice_at_0.50", "hard_dice_at_{:.2f}".format(0.50))
vox_col  = pick("voxels_clean", "lesion_voxels", "mask_voxels_clean")

soft  = col(soft_col)
hard  = col(hard_col)
vox   = col(vox_col, float)
hfe   = col("hi_freq_energy")
lapv  = col("lap_var")
score = col("score")

# Ensure positive sizes for log
logv  = np.log10(np.where(vox > 0, vox, np.nan))

# --- helpers ---
def nanmean(x): 
    m = ~np.isnan(x)
    return float(np.mean(x[m])) if m.any() else float("nan")

def nan_corr(x, y):
    m = (~np.isnan(x)) & (~np.isnan(y))
    if m.sum() < 3: return float("nan")
    return float(np.corrcoef(x[m], y[m])[0,1])

def partial_corr(y, x, control):
    m = (~np.isnan(y)) & (~np.isnan(x)) & (~np.isnan(control))
    if m.sum() < 5: return float("nan")
    y, x, c = y[m], x[m], control[m]
    A = np.vstack([c, np.ones_like(c)]).T
    by, *_ = np.linalg.lstsq(A, y, rcond=None)
    bx, *_ = np.linalg.lstsq(A, x, rcond=None)
    ry = y - A @ by
    rx = x - A @ bx
    return nan_corr(ry, rx)

# --- summary stats ---
macro_soft = nanmean(soft)
macro_hard = nanmean(hard)

r_size     = nan_corr(soft, logv)
r_hfe      = nan_corr(soft, hfe)
r_lapv     = nan_corr(soft, lapv)
r_score    = nan_corr(soft, score)

# partials controlling for lesion size
pr_hfe     = partial_corr(soft, hfe, logv)
pr_lapv    = partial_corr(soft, lapv, logv)
pr_score   = partial_corr(soft, score, logv)

# size-bin summary
bins   = [0, 2500, 10000, 30000, 60000, 10**9]
labels = ["[0,2.5k)", "[2.5k,10k)", "[10k,30k)", "[30k,60k)", "[60k,+inf)"]
bin_lines = []
for i in range(len(bins)-1):
    lo, hi = bins[i], bins[i+1]
    m = (vox >= lo) & (vox < hi)
    n = int(np.sum(m))
    bin_soft = nanmean(soft[m]) if n>0 else float("nan")
    bin_hard = nanmean(hard[m]) if n>0 else float("nan")
    bin_lines.append(f"- **{labels[i]}**: n={n}, mean soft={bin_soft:.4f}, mean hard={bin_hard:.4f}")

# cohort split by leading letter in key (if present)
keys = [r.get("key","") for r in rows]
lead = np.array([(k[0].lower() if k else "") for k in keys])
summary_lines = []
for tag in sorted(set(lead) - {""}):
    m = (lead == tag)
    summary_lines.append(f"- **{tag}** | n={int(np.sum(m))} | mean soft={nanmean(soft[m]):.4f}")

# --- compose Markdown ---
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_MD = EVAL_DIR / f"evaluation_summary_{ts}.md"

md = []
md.append("# Held-out Evaluation Summary")
md.append("")
md.append(f"**Run:** `{RUN_DIR.name}`")
md.append(f"**Per-case macro Dice (soft):** {macro_soft:.4f}")
md.append(f"**Per-case macro Dice (hard @0.50):** {macro_hard:.4f}")
md.append("")
md.append("## Correlations")
md.append(f"- r(soft Dice, log10 lesion size): **{r_size:.3f}**")
md.append(f"- r(soft Dice, hi_freq_energy): **{r_hfe:.3f}**  | partial (| size): **{pr_hfe:.3f}**")
md.append(f"- r(soft Dice, lap_var): **{r_lapv:.3f}**       | partial (| size): **{pr_lapv:.3f}**")
md.append(f"- r(soft Dice, score): **{r_score:.3f}**        | partial (| size): **{pr_score:.3f}**")
md.append("")
md.append("## Mean Dice by lesion-size bin")
md.extend(bin_lines)
if summary_lines:
    md.append("")
    md.append("## Cohort summary (by key prefix)")
    md.extend(summary_lines)

# Optionally embed figures if they exist
figs = [
    "scatter_soft_vs_logvox.png",
    "scatter_soft_vs_hfe.png",
    "scatter_soft_vs_lapvar.png",
    "scatter_soft_vs_score.png",
]
existing_figs = [p for p in figs if (FIG_DIR / p).exists()]
if existing_figs:
    md.append("")
    md.append("## Figures")
    for p in existing_figs:
        md.append(f"- ![{p}]({FIG_DIR.name}/{p})")

md.append("")
md.append("## Sources")
md.append(f"- Joined per-case CSV: `{CSV_JOIN}`")
md.append(f"- Figures dir (if generated): `{FIG_DIR}`")

OUT_MD.write_text("\n".join(md))
print("Wrote:", OUT_MD)


Using eval CSV: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3/runs/20251110_101813/test_eval/test_metrics_with_manifest_FIXED_20251110_160551.csv
Wrote: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3/runs/20251110_101813/test_eval/evaluation_summary_20251110_161303.md
